In [ ]:
import requests
import pandas as pd
import time
import io
from typing import List, Dict, Any, Tuple
import json
from tqdm import tqdm

In [ ]:
district_id_map = {
    "002": "24 Parganas(S)",
    "001": "24 Parganas(N)",
    "003": "Bankura",
    "005": "Burdwan",
    "004": "Birbhum",
    "027": "Burdwan(E)",
    "026": "Burdwan(W)",
    "006": "Coochbehar",
    "007": "Darjeeling",
    "008": "Dinajpur(N)",
    "009": "Dinajpur(S)",
    "010": "Hooghly",
    "011": "Howrah",
    "012": "Jalpaiguri",
    "021": "Jhargram",
    "022": "Kalimpong",
    "013": "Kolkata",
    "014": "Maldah",
    "015": "Medinipore(E)",
    "016": "Medinipore(W)",
    "017": "Murshidabad",
    "018": "Nadia",
    "025": "Others",
    "019": "Purulia"
    }
df_dist_id_maps = pd.DataFrame(district_id_map, index=[0])
df_dist_id_maps.to_csv("../datasets/id_dist_mapping.csv",encoding= 'utf_8', index=False)

In [ ]:
main_url = "http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp"
ajax_url = "http://emis.wbpcb.gov.in/airquality/JSP/aq/fetch_val_ajax.jsp" 

In [ ]:
from json import JSONDecodeError


def station_grabber(dist_id: str, main_url: str, ajax_url: str) -> Dict[str, str]:
    all_stations : Dict[str, str] = {}

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{dist_id}...")
        
        payload = {
            'id': dist_id,
            'type': 'stationM'
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        try:
            data = response.json()
            station_list = data.get("list", [])
            for stn in station_list:
                all_stations[stn['name']] = stn['id']
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {dist_id}. Server said: {response.text[:100]}")
            
    time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return all_stations

station_datas : Dict[str, Any] = {}
for dist in district_id_map.keys():
    stn_under_dist = station_grabber(dist, main_url=main_url, ajax_url=ajax_url)
    station_datas[dist] = stn_under_dist

In [ ]:
all_station_list_vanilla = [x for dict in station_datas.values() for x in dict.values()]

In [ ]:
def date_availability_of_station(stn_code: str, main_url: str, ajax_url: str) -> List:
    all_available_dates : List = []

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{stn_code}...")
        
        payload = {
            'stn_code': stn_code,
            'type': 'date'
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        
        try:
            data = response.json()
            date_list = data.get("list", [])
            
            all_available_dates = [list(x.values())[0] for x in date_list]
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
    time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return all_available_dates

#print(date_availability_of_station("st30", main_url, ajax_url))
all_available_dates : dict[str, List] = {}
for id in all_station_list_vanilla:
    all_available_dates[id] = date_availability_of_station(id,main_url, ajax_url)

In [ ]:
def date_availability_of_station(stn_code: str, date: str, main_url: str, ajax_url: str) -> Dict[str, Any]:
    aqi_dict : Dict[str, Any] = {} 

    with requests.Session() as session:
        print("Grabbing session cookies...")
        session.get(main_url, headers={"User-Agent": "Mozilla/5.0"})
        
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': main_url
        }
        
        print(f"Fetching station maps for district_{stn_code}...")
        
        payload = {
            'stn_code': stn_code,
            'date': date,
            'type': "aqi"
        }
        
        response = session.post(ajax_url, data=payload, headers= headers)
        
        try:
            data = response.json()
            aqi_data_specific = data.get("list", [])

            for param_dict in aqi_data_specific:
                aqi_dict[param_dict['pname']] = float(param_dict['value'])
        except JSONDecodeError:
            print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
    #time.sleep(0.5) #wanna be polite to the server 🙏🏿
    return aqi_dict

date_availability_of_station('st67', "25/02/2026", main_url, ajax_url)

In [ ]:
def process_single_station(stn_code:str, date_list:List[str]) -> Tuple[str, List[Dict[str, Any]]]:
    station_results: List[Dict[str, Any]] = []
    
    
    with requests.Session() as session:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:149.0) Gecko/20100101 Firefox/149.0',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'http://emis.wbpcb.gov.in',
            'Referer': "http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp"
        }
        
        session.get("http://emis.wbpcb.gov.in/airquality/JSP/aq/filter_for_aqiM.jsp", headers={"User-Agent": "Mozilla/5.0"})
        
        for date_str in tqdm(date_list, desc=f"Processing for station {stn_code}"):
            aqi_dict: Dict[str, Any] = {}
            
            m, d, y = date_str.split('/') 
            date_m = f"{d}/{m}/{y}"
            payload = {
                    'stn_code': stn_code,
                    'date': date_m,
                    'type': "aqi"
                }
            response = session.post("http://emis.wbpcb.gov.in/airquality/JSP/aq/fetch_val_ajax.jsp" , data=payload, headers= headers)
            try:
                
                data = response.json()
                aqi_data_specific = data.get("list", [])

                for param_dict in aqi_data_specific:
                    aqi_dict[param_dict['pname']] = float(param_dict['value'])
            except JSONDecodeError:
                print(f"  [!] Failed to parse district {stn_code}. Server said: {response.text[:100]}")
            
            aqi_dict['date'] = date_m 
            station_results.append(aqi_dict)
    
    return stn_code, station_results

In [ ]:
aqi_data_acquired: Dict[str, List[Dict[str, Any]]] = {}

for stn, date_list in all_available_dates.items():
    _, station_results = process_single_station(stn, date_list)
    aqi_data_acquired[stn] = station_results

In [ ]:
with open("../datasets/aqi_data.json", "w") as file:
    json.dump(aqi_data_acquired, file)